# 📘 CurriculumMD: PDF to Markdown Extraction with GPU
### สกัดตารางและข้อความหลักสูตรแกนกลาง (8 กลุ่มสาระ) ด้วย Marker / MinerU บน Google Colab GPU

**สถาปัตยกรรมกระบวนการ:**
1. อ่านไฟล์ PDF จาก `Google Drive/CurriculumMD/input_pdfs`
2. รัน AI Deep Learning Model (`Marker` / `MinerU`) บน GPU ฟรีของ Colab
3. บันทึกผลลัพธ์เป็น Clean Markdown กลับลง `Google Drive/CurriculumMD/output_markdown` อัตโนมัติ

> 💡 **คำแนะนำ:** ไปที่เมนู **Runtime > Change runtime type** แล้วเลือกฮาร์ดแวร์เป็น **T4 GPU** (หรือ L4/A100 หากมี Colab Pro)

## ⚙️ ขั้นตอนที่ 1: ตรวจสอบ GPU และเชื่อมต่อ Google Drive

In [ ]:
# 1. เช็คความพร้อมของ GPU
!nvidia-smi

# 2. เชื่อมต่อ Google Drive (โฟลเดอร์จะซิงค์กับเครื่อง Desktop ทันที)
from google.colab import drive
import os

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/CurriculumMD'
INPUT_DIR = os.path.join(BASE_DIR, 'input_pdfs')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output_markdown')

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\n✅ เชื่อมต่อ Google Drive สำเร็จ!")
print(f"📂 Input PDF Folder : {INPUT_DIR}")
print(f"📂 Output MD Folder : {OUTPUT_DIR}")

## 📦 ขั้นตอนที่ 2: ติดตั้งโมเดลสกัดเอกสาร (Marker-PDF)

In [ ]:
# ติดตั้ง marker-pdf (โมเดลแปลง PDF เป็น Markdown คุณภาพสูง รองรับภาษาไทย)
!pip install -q marker-pdf
print("✅ ติดตั้ง Marker-PDF เรียบร้อยพร้อมใช้งาน!")

## 🔍 ขั้นตอนที่ 3: ตรวจสอบรายการไฟล์ PDF ที่พร้อมสกัด

In [ ]:
import glob

pdf_files = glob.glob(f"{INPUT_DIR}/*.pdf")
print(f"พบไฟล์ PDF ในโฟลเดอร์ทั้งหมด {len(pdf_files)} ไฟล์:\n")
for i, f in enumerate(pdf_files, 1):
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f" {i}. {os.path.basename(f)} ({size_mb:.2f} MB)")

if not pdf_files:
    print("⚠️ ยังไม่พบไฟล์ PDF ในโฟลเดอร์ กรุณาคัดลอกไฟล์ใส่โฟลเดอร์ Google Drive/CurriculumMD/input_pdfs ในเครื่องของคุณ")

## 🚀 ขั้นตอนที่ 4: เริ่มต้นสกัด PDF ทั้งหมดเป็น Clean Markdown (Batch Processing)

In [ ]:
import subprocess
import time

pdf_files = glob.glob(f"{INPUT_DIR}/*.pdf")

start_total = time.time()
print(f"🚀 เริ่มต้นการสกัดข้อมูลจำนวน {len(pdf_files)} ไฟล์...\n")

for idx, pdf_path in enumerate(pdf_files, 1):
    filename = os.path.basename(pdf_path)
    base_name = os.path.splitext(filename)[0]
    target_folder = os.path.join(OUTPUT_DIR, base_name)
    os.makedirs(target_folder, exist_ok=True)
    
    print(f"[{idx}/{len(pdf_files)}] ⏳ กำลังประมวลผล: {filename}...")
    t0 = time.time()
    
    # รันคำสั่ง marker_single พร้อมรองรับภาษาไทย
    cmd = [
        "marker_single",
        pdf_path,
        target_folder,
        "--batch_multiplier", "2",
        "--langs", "Thai,English"
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - t0
    
    if result.returncode == 0:
        print(f"   ✅ สำเร็จใน {elapsed:.1f} วินาที! บันทึกลง: {target_folder}")
    else:
        print(f"   ❌ เกิดข้อผิดพลาด:")
        print(result.stderr[:500])

total_elapsed = time.time() - start_total
print(f"\n🎉 ประมวลผลเสร็จสิ้นทุกไฟล์ รวมเวลา: {total_elapsed/60:.2f} นาที!")

## 📊 ขั้นตอนที่ 5: สรุปผลลัพธ์ไฟล์ Markdown ใน Google Drive

In [ ]:
md_results = glob.glob(f"{OUTPUT_DIR}/**/*.md", recursive=True)
print(f"📄 พบไฟล์ Markdown ที่สร้างสำเร็จทั้งหมด: {len(md_results)} ไฟล์\n")
for md in md_results:
    size_kb = os.path.getsize(md) / 1024
    rel = os.path.relpath(md, OUTPUT_DIR)
    print(f" - {rel} ({size_kb:.1f} KB)")

print("\n💡 ข้อมูลจะถูกซิงค์กลับมาที่คอมพิวเตอร์ของคุณอัตโนมัติผ่าน Google Drive for Desktop")